In [ ]:
from langchain.agents import AgentExecutor, create_tool_calling_agent, tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel


In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# ReAct 스타일 vs. tool_calls 스타일

In [ ]:
# 언어 모델의 출력 - 도구 호출 기능이 있는 언어 모델

# 도구 호출 기능이 있는 언어 모델은 텍스트가 아닌 구조화된 JSON-like 형태를 만들어냄
# 도구 호출 기능이 있는 언어 모델은 프롬프트로 어떤 규칙을(Action:, Action Input:) 유도하지 않음

# 1. 도구 호출이 필요 없는 경우에는 tool_calls가 비어 있음
# {
#   "content": "파리의 에펠탑은 1889년에 건설되었다.",
#   "tool_calls": []
# }
# 이 경우, Agent는 AgentFinish로 변경

# 2. 도구 호출을 해야 하는 경우에는 tool_calls 안에 함수명(name)과 인자(arguments)가 들어있음
# {
#   "content": null,
#   "tool_calls": [
#     {
#       "name": "Search",
#       "arguments": {
#         "query": "에펠탑 최신 방문객 수"
#       }
#     }
#   ]
# }
# 이 경우, Agent는 AgentAction으로 변경
# -----------------------------------------------------------------------------------------------

# role 키는 Chat API 기반 모델이면 있음
# Chat API 기반 모델이란 OpenAI가 제공하는 대화 형식을 따르는 모델

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

llm = ChatOpenAI(model="gpt-4o-mini")

@tool
def multiply(a: int, b: int) -> int:
    """두 수를 곱한다"""
    return a * b

tools = [multiply]

llm_with_tools = llm.bind_tools(tools)
# llm.bind_tools(tools)를 통해 Agent/AgentExecutor 없이도 모델이 직접 도구 호출을 할 수 있음

result = llm_with_tools.invoke("5 곱하기 3 해줘")
print(result.tool_calls)
# tool_calls는 호출하라고 지시한 도구 정보

# result.tool_calls
[
    {
        'name': 'multiply', 
        'args': {'a': 5, 'b': 3}, 
        'id': 'call_5wh8TyHCthCPaUSXxOE1d8FO', 
        'type': 'tool_call'
    }
]

In [ ]:
print(type(result))
for k, v in result.model_dump().items():
    print(k, '---', v)

In [ ]:
# tool_calls가 두 번 등장함

# 1. AIMessage.tool_calls는 LangChain이 언어 모델의 응답을 파싱해서 정리한 결과
# ---> 언어 모델의 출력을 파싱하는 주체는 LLM 내부의 wrapper가 담당

# 2. AIMessage.additional_kwargs["tool_calls"]는 언어 모델이 출력한 원본

# LangChain 내부 로직에서 실제로 사용하는 쪽은 AIMessage.tool_calls

In [ ]:
# content 필드는 원칙적으로 모델이 자연어 텍스트를 만들어내면 그 값으로 채워짐

# 1. 모델이 도구 호출만 출력한 경우 'content' 필드는 비어있고 tool_calls만 출력
# 2. 모델이 도구 호출과 함께 자연어도 함께 출력하는 경우 'content' 필드에 자연어 텍스트와 함께 tool_calls를 출력

# 기준
# 1. 어떻게 학습 또는 튜닝을 했는가?
# 2. 프롬프트에서 “도구 호출만 하고 아무 말도 하지 마”라고 한 경우에는 tool_calls만 출력
# 3. 모델별 차이


# ★★★★★★★★★★
# 언어 모델이 최종 답을 찾은 경우에는 tool_calls를 만들지 않고 content에 자연어 텍스트를 채워서 반환
# 이 경우 Agent는 AgentFinish로 변환

In [ ]:
# ReAct 스타일 vs. tool_calls 스타일

In [ ]:
# 프롬프트 생성 단계
# 1. 사용자 입력은 "5 더하기 3 해줘" 
# 2. agent_executor.invoke({"input": "5 더하기 3 해줘"})를 통해 Agent에 넘김
# 3. Agent는 ChatPromptTemplate를 준비하고 있음
#    이 프롬프트에는 사용자 질문에 해당하는 {input}, intermediate_steps로 채워질 {agent_scratchpad}가 있음
#    당연히 최초 실행 시 intermediate_steps는 []임
# 4. Agent는 다음과 같은 메시지 리스트를 만들어 냄

[
  SystemMessage("당신은 훌륭한 AI 어시스턴트입니다."),
  HumanMessage("5 더하기 3 해줘"),
]

# 5. Agent가 언어 모델에 프롬프트를 전달, 언어 모델은 응답으로 tool_calls를 반환
# 6. tool_calls를 출력하면 Agent는 파싱을 해서 AgentAction/AgentFinish 둘 중의 하나로 변환
#    이 과정에서 파싱은 Agent가 담당

# 7. Agent는 AgentAction/AgentFinish 둘 중의 하나를 AgentExecutor에 전달하고 
# 8. AgentExecutor는 AgentAction인 경우 해당 도구를 호출하고 그 결과를 함께 intermediate_steps에 저장하고 이를 다시 Agent에게 전달
# 9. Agent는 전달받은 intermediate_steps에서 
#    AgentAction은 AIMessage로, 
#    함수 호출의 결과는 FunctionMessage로 바꾸어서 scratch_pad에 집어 넣는데
#    ChatPromptTemplate + AIMessage + FunctionMessage의 형태로 ChatPromptTemplate에 덧 붙임
#    문제는 ChatPromptTemplate에 'input' 변수가 있는데 사용자의 최초 입력에 해당하는 변수임
#    이 때문에 AgentExecutor는 항상 AgentExecutor.invoke({"input": "..."})를 호출하는 것임

In [ ]:
# "도구 호출 기능"이 있는 언어 모델과 상대하는 에이전트 생성
# create_openai_tools_agent와 create_tool_calling_agent

In [ ]:
# create_openai_tools_agent와 create_tool_calling_agent의 차이점

# 공통점은 Tool Calling 기능을 사용하는 에이전트를 생성한다는 점
# 차이점은

# create_openai_tools_agent는 OpenAI 모델에 특화되어 있음
# 특화되었다는 의미
# 1. create_openai_tools_agent는 OpenAI의 응답 구조를 가장 효율적으로 파싱
# 2. @tools로 정의한 파이썬 함수를 OpenAI가 요구하는 JSON 스키마로 정확하게 변환

# create_tool_calling_agent: 언어 모델이 tool_calls 형식을 지원하기만 하면 어떤 모델이든 사용 가능하도록 설계

# 그래서?
# OpenAI 모델을 사용 > create_openai_tools_agent가 가장 나은 선택
# OpenAI 외의 다른 모델을 사용하거나, 모델 변경을 염두에 둔 범용적인 코드를 작성 > create_tool_calling_agent를 선택

In [ ]:
# create_openai_tools_agent - gemini-2.5-flash인 경우

from langchain.agents import create_tool_calling_agent
from langchain.agents import create_openai_tools_agent

@tool
def add_numbers(a: int, b: int) -> int:
    """두 개의 정수를 더합니다."""
    return a + b

@tool
def multiply_numbers(a: int, b: int) -> int:
    """두 개의 정수를 곱합니다."""
    return a * b

# OpenAI와 Google Gemini의 출력
# llm = ChatOpenAI(model="gpt-4o", temperature=0)
llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash")

tools = [add_numbers, multiply_numbers]

# 일반적인 대화 프롬프트를 사용
prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", "너는 사용자 질문에 답변하는 훌륭한 조수야. 주어진 도구들을 활용해서 답변을 생성해."),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}"),
    ]
)

agent = create_openai_tools_agent(
    llm=llm,
    tools=tools,
    prompt=prompt_template
)

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

response = agent_executor.invoke({"input": "5 더하기 3을 한 다음에 그 결과에 2를 곱해줘."})

print(f"\n최종 답변: {response['output']}")

# 누가 만들어낸 결과? response는 agent_executor의 결과, 'output' 속성에 최종 결과
# chain = prompt | llm
# response = chain.invoke({'input': ...})
# response.content

In [ ]:
# create_openai_tools_agent - gpt-4o인 경우

from langchain.agents import create_tool_calling_agent
from langchain.agents import create_openai_tools_agent

@tool
def add_numbers(a: int, b: int) -> int:
    """두 개의 정수를 더합니다."""
    return a + b

@tool
def multiply_numbers(a: int, b: int) -> int:
    """두 개의 정수를 곱합니다."""
    return a * b

# OpenAI와 Google Gemini의 출력
llm = ChatOpenAI(model="gpt-4o", temperature=0)
# llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash")

tools = [add_numbers, multiply_numbers]

# 일반적인 대화 프롬프트를 사용
prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", "너는 사용자 질문에 답변하는 훌륭한 조수야. 주어진 도구들을 활용해서 답변을 생성해."),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}"),
    ]
)

agent = create_openai_tools_agent(
    llm=llm,
    tools=tools,
    prompt=prompt_template
)

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

response = agent_executor.invoke({"input": "5 더하기 3을 한 다음에 그 결과에 2를 곱해줘."})

print(f"\n최종 답변: {response['output']}")

In [ ]:
# > Entering new AgentExecutor chain...
# 
# Invoking: `add_numbers` with `{'a': 5, 'b': 3}`
# 
# 
# 8
# Invoking: `multiply_numbers` with `{'a': 8, 'b': 2}`
# 
# 
# 165 더하기 3은 8이고, 그 결과에 2를 곱하면 16입니다.
# 
# > Finished chain.

# 최종 답변: 5 더하기 3은 8이고, 그 결과에 2를 곱하면 16입니다.
# --------------------------------------------------------------------------------
# Invoking: `add_numbers` with `{'a': 5, 'b': 3}`는 
# AgentExecutor가 도구 호출을 하면서 만든 문자열
# 8은 그 결과이고, 내부적으로 이 둘을 사용해서 intermediate_steps에 기록
# 
# Invoking: `multiply_numbers` with `{'a': 8, 'b': 2}`는 
# AgentExecutor가 도구 호출을 하면서 만든 문자열
# 16은 그 결과이고, 내부적으로 이 둘을 intermediate_steps에 기록

# 이를 언어모델에게 전달하면 언어모델이 (최종 판단이라고 결정을 해서) tool_calls 없이 
# content에 넣어서 전달한 문자열이 
# "5 더하기 3은 8이고, 그 결과에 2를 곱하면 16입니다."

# 모델에 상관없이 동일한 이유는 언어 모델이 만든 출력이 아니라
# LLM의 외부에서 작동하는 LangChain의 AgentExecutor가 생성했기 때문
# 모델에 따라 달라진 부분은 최종 결과에 언어 모델이 덧붙인 자연어 텍스트임

# gemini-2.5-flash인 경우
# 최종 답변: 5 더하기 3을 한 다음에 그 결과에 2를 곱하면 16이 됩니다.

# gpt-4o인 경우
# 최종 답변: 5 더하기 3은 8이고, 그 결과에 2를 곱하면 16입니다.

In [ ]:
# create_tool_calling_agent

@tool
def add_numbers(a: int, b: int) -> int:
    """두 개의 정수를 더합니다."""
    return a + b

@tool
def multiply_numbers(a: int, b: int) -> int:
    """두 개의 정수를 곱합니다."""
    return a * b


llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash")
tools = [add_numbers, multiply_numbers]

prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", "너는 사용자 질문에 답변하는 훌륭한 조수야. 주어진 도구들을 활용해서 답변을 생성해."),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}"),
    ]
)

agent = create_tool_calling_agent(
    llm=llm,
    tools=tools,
    prompt=prompt_template
)

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

response = agent_executor.invoke({"input": "5 더하기 3을 한 다음에 그 결과에 2를 곱해줘."})

print(f"\n최종 답변: {response['output']}")

In [ ]:
# 동일한 사용자 요청 {"input": "5 더하기 3을 한 다음에 그 결과에 2를 곱해줘."}에 대해서
# ReAct 스타일로 진행한 경우 에러가 발생했음

# ReAct 스타일은 자연어 텍스트로 추론하며 이는 초기 언어 모델의 능력인 자연어 생성 능력을 활용하자는 의도였음
# Tool Calling 스타일은 구조화된 JSON을 생성할 수 있음, 자연어 생성 능력 뿐만 아니라 특정 형식에
# 맞춰서 출력하도록 언어 모델을 학습 시켰기 때문

# ReAct 스타일로 진행한 경우 에러 > 언어 모델의 문제라기 보다는 파서의 능력
# JSON 형태의 파싱은 너무나 쉬운 반면에 ReAct 스타일의 언어 모델의 출력인 자연어 텍스트 파싱은 쉬운 문제가 아님

In [ ]:
@tool
def add_two(x: int) -> int:
    """입력값에 2를 더합니다."""
    return x + 2

tools = [add_two]

# create_tool_calling_agent는 ChatPromptTemplate를 사용하여 에이전트의 프롬프트 템플릿을 만듬
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 유용한 도구를 사용할 수 있는 똑똑한 에이전트입니다."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

llm=ChatGoogleGenerativeAI(model="gemini-2.0-flash")

agent = create_tool_calling_agent(llm=llm, tools=tools, prompt=prompt)

agent_executor = AgentExecutor(
    # agent는 반복 실행
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True
)

result = agent_executor.invoke({
    "input": "3에 2를 더하면 얼마야?",
    "chat_history": []
})
print(result)

# 프롬프트 내부의 MessagesPlaceholder(variable_name="chat_history")에 대화 이력을 추가하려면
# HumanMessage와 AIMessage 객체를 사용하여 대화 기록을 만들고 invoke 메서드에 명시적으로 전달해야 함
# agent_executor.invoke({
#     "input": "3에 2를 더하면 얼마야?",
#     "chat_history": chat_history # 이전 대화 기록을 명시적으로 전달
# })

# chat_history = [
#     HumanMessage(content="내 이름은 김민준이야."),
#     AIMessage(content="안녕하세요, 김민준님! 무엇을 도와드릴까요?"),
#     HumanMessage(content="어제 봤던 그 영화 제목이 뭐였지?"),
#     AIMessage(content="어제는 영화에 대해 이야기하지 않았습니다.")
# ]

# history
# chat_history + agent_scratchpad가 agent_scratchpad만 사용하는 것 보다 언어 모델에게 문제 해결을 더 잘하게 할 수 있다

In [ ]:

@tool
def summarize(text: str) -> str:
    """긴 문장을 간단히 요약합니다."""
    return text[:100] + "..."

@tool
def extract_keywords(text: str) -> list:
    """문장에서 핵심 키워드를 추출합니다."""
    return [word for word in text.split() if len(word) > 5][:5]
    # text를 공백으로 분리해서 단어 리스트를 만든 다음, 단어의 길이가 5보다 크면 리스트에 저장한 다음 첫 다섯개만 리턴

@tool
def add_days(days: int) -> str:
    """오늘 날짜에서 N일을 더한 날짜를 반환합니다."""
    from datetime import datetime, timedelta
    return (datetime.now() + timedelta(days=days)).strftime("%Y-%m-%d")

@tool
def multiply(x: int, y: int) -> int:
    """두 숫자를 곱합니다."""
    return x * y

tools = [summarize, extract_keywords, add_days, multiply]

prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 다양한 도구를 사용할 수 있는 유능한 에이전트입니다."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

llm=ChatGoogleGenerativeAI(model="gemini-2.0-flash")

agent = create_tool_calling_agent(llm=llm, tools=tools, prompt=prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True
)

queries = [
    "다음 문장을 요약해줘: LangChain은 다양한 LLM을 연결하여 복잡한 작업을 수행할 수 있는 프레임워크입니다.",
    "이 문장에서 핵심 키워드를 뽑아줘: LangChain은 체인, 에이전트, 메모리 등을 통해 복잡한 작업을 수행합니다.", 
    "오늘부터 10일 뒤 날짜가 뭐야?",
    "23과 7을 곱하면 얼마야?"
]

for q in queries:
    result = agent_executor.invoke({"input": q, "chat_history": []})
print(f"\n질문: {q}\n응답: {result['output']}")


In [ ]:
# ★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
# 언어 모델이 만든 tool_calls의 내용을 파싱/검증은 Agent의 몫
# ------------------------------------------------------------------------
class TranslationRequest(BaseModel):
    text: str
    target_language: str

@tool
def translate(req: TranslationRequest) -> str:
    """입력된 문장을 지정된 언어로 번역합니다."""
    return f"'{req.text}' → [{req.target_language}] 언어로 번역됨 (예시)"

# Agent는 create_tool_calling_agent 함수를 통해 tools 목록을 전달받을 때
# 도구들의 args_schema를 자동으로 파악 (args_schema는 도구 함수가 받을 입력값의 설계도 역할)

# args_schema -> 도구 스키마

# 1.
# @tool 데코레이터를 함수 위에 붙일 때, 함수에 타입 힌트가 있으면
# args_schema를 자동으로 추가
# req: TranslationRequest을 보고 args_schema=TranslationRequest를 자동으로 추론

# 2. args_schema를 자동으로 추론한 다음 단계는 도구 스키마를 생성
# 도구 스키마란 도구 사용 설명서임
# {
#   "name": "translate",
#   "description": "입력된 문장을 지정된 언어로 번역합니다.",
#   "parameters": {
#     "type": "object",
#     "properties": {
#       "text": {
#         "type": "string",
#         "description": "번역할 원문"
#       },
#       "target_language": {
#         "type": "string",
#         "description": "번역을 원하는 언어"
#       }
#     },
#     "required": ["text", "target_language"]
#   }
# }

# 3. 도구 스키마를 만들면 언어 모델에 이 도구 스키마를 전달
#    - 언어 모델은 에이전트가 어떤 도구들을 사용할 수 있는지 알게 되고
#    - 도구가 어떤 작업을 하는지(description) 이해
#    - 도구를 호출할 때 어떤 인수(arguments)가 필요한지 정확히 파악

# 4. 언어 모델이 tool_calls 객체를 출력
# {
#   "tool_calls": [
#     {
#       "function": {
#         "name": "translate",
#         "arguments": {
#           "text": "안녕하세요, 오늘 날씨가 참 좋네요.",
#           "target_language": "영어"
#         }
#       }
#     }
#   ]
# }

# 5. Agent는 tool_calls를 파싱하는 과정에서 필드 유효성 검증을 실시
#    tool_calls의 arguments 딕셔너리에 있는 키와 값을 사용하여 TranslationRequest Pydantic 모델을 인스턴스화하려고 시도
#    - "text": "안녕하세요, 오늘 날씨가 참 좋네요.",
#    - "target_language": "영어"
#    text나 target_language 필드가 누락되었거나 str 타입이 아니면, Pydantic은 즉시 오류를 발생

tools = [translate]
# ------------------------------------------------------------------------

prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 번역 도구를 사용할 수 있는 유능한 에이전트입니다."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

llm=ChatGoogleGenerativeAI(model="gemini-2.0-flash")

agent = create_tool_calling_agent(llm=llm, tools=tools, prompt=prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True
)

# 🧪 실행 예시

result = agent_executor.invoke({
    "input": "이 문장을 영어로 번역해줘: 안녕하세요, 오늘 날씨가 참 좋네요.",
    "chat_history": []
})
print(f"\n응답: {result['output']}")


In [ ]:
tool_object = tools[0]
print(tool_object.args_schema.model_json_schema())

In [ ]:
# result = agent_executor.invoke()에서
# result의 속성

print(result['input']) # agent_executor에 처음 전달된 최초 사용자 질문
print(result['output']) # 에이전트가 최종적으로 생성한 답변
print(result['chat_history']) # MessagesPlaceholder(variable_name="chat_history") 내용을 출력하는 것

In [ ]:
# eval
# eval(expr)은 문자열로 되어 있는 expr을 파이썬 코드라고 생각하고 실행

# eval(A, B)은 두 번째 인자로 딕셔너리를 받아, 그 딕셔너리에 정의된 변수와 함수만을 사용하여 코드를 실행

# safe_vars = {'x': 5, 'y': 10}
# result = eval("x + y", safe_vars)

# a = 100
# b = 200
# result = eval("a + b", safe_vars) >>> NameError

# safe_globals = {'__builtins__': None, 'math': math}
# result = eval("math.sqrt(16)", safe_globals)
# 'math'라는 이름 >>> safe_globals 딕셔너리에서 'math' 키를 찾음 >>> math' 키에 매핑된 실제 math 모듈을 참조하여 math.sqrt(16)을 실행

In [ ]:
# pydantic + tools 변수 동적으로 추가하기 + 문제에 적합한 프롬프트의 중요성

from pydantic import BaseModel, Field
import math

class CalcInput(BaseModel):
    expr: str = Field(..., description="계산할 수식, 예: 'math.sqrt(16)' 또는 '2*(3+4)'")

# 함수가 받는 인자는 pydantic 모델 CalcInput에서 정의한대로 전달해야 함
# args_schema는 함수 인자에 관련된 스키마
@tool(args_schema=CalcInput)
def calc(expr: str) -> str:
    """수학 수식을 계산합니다. math 모듈만 허용됩니다."""
    safe_globals = {"__builtins__": {}, "math": math}
    # eval('__import__("os").system("rm -rf /")')는 모든 파일 삭제
    # __builtins__: {} >>> eval() 함수는 내장 함수에 접근을 못 하게 됨
    # 내장 함수는 import 없이 실행할 수 있는 함수
    # 문제는 __import__("os")도 내장 함수라는 점 >>> os 모듈을 불러올 수 있음
    try:
        result = eval(expr, safe_globals)
        # eval()은 문자열로 된 파이썬 표현식(expression)을 실행하고 그 결과값을 반환
        # eval("2 + 3")는 5
        # eval(expr, A): A에서 정의한 변수와 함수만 사용해서 expr을 평가하라는 의미
        return str(result)
    except Exception as e:
        return f"계산 오류: {str(e)}"

tools = [calc]

base_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "사용자가 제공한 모든 수식에 대해, 네가 직접 계산하지 말고 무조건 calc 도구를 사용해서 결과를 얻어."
     "너는 계산 도구를 사용할 수 있는 AI야. 사용자가 수식을 입력하면 calc 도구를 호출해서 결과를 얻고, "
     "그 결과를 바탕으로 친절하고 이해하기 쉬운 설명을 만들어줘. 숫자만 던지지 말고, 왜 그런 결과가 나왔는지도 알려줘."),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# 도구 이름과 설명을 문자열로 렌더링
def render_tools(ts):
    return "\n".join(f"- {t.name}: {t.description}" for t in ts)


prompt = base_prompt.partial(tools=render_tools(tools))
# 프롬프트에 도구 정보를 전달하는 방법
# tools라는 키로 render_tools(tools)를 값으로 프롬프트에 추가
# base_prompt에 tools 변수는 없지만
# partial 메서드를 통해 동적으로 추가되면, LangChain은 이 정보를 언어 모델이 도구 호출에 사용할 수 있도록 자동으로 처리
# 결론: tools 변수는 LangChain의 에이전트 프레임워크가 인식하는 특별한 키워드

print(prompt.input_variables)
print(prompt.partial_variables)


In [ ]:

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)

agent = create_tool_calling_agent(llm=llm, tools=tools, prompt=prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=5
)

In [ ]:

# 🧪 6. 실행 예시
queries = [
    "math.sqrt(144)는 뭐야?",
    "math.sin(math.pi / 2)의 값은?",
    "2*(3+4)는 어떻게 계산돼?",
]

for q in queries:
    result = agent_executor.invoke({"input": q, "chat_history": []})
    print(f"질문: {q}, 응답: {result['output']}")


In [ ]:
safe_globals = {"__builtins__": None, "math": math}
result = eval("2*(3+4)", safe_globals)
print(result)